<div style="border-radius: 10px; padding: 32px 0px; border: 1px solid rgba(128,128,128,0.2);">

  <div style="display: flex; justify-content: space-between; align-items: flex-start; flex-wrap: wrap; gap: 16px; padding: 0px 32px;">

  <div>
    <div style="font-size: 0.75rem; letter-spacing: 3px; text-transform: uppercase; font-weight: 600; margin-bottom: 10px; opacity: 0.6;">
      Máster Universitario en Big Data y Computación en la Nube.
    </div>
    <div style="font-size: 1.5rem; font-weight: 700; margin-bottom: 4px;">
      Trabajo de Fin de Máster
    </div>
    <div style="font-size: 1rem; font-weight: 400; opacity: 0.75;">
      Clasificador taxonómico de boletines oficiales españoles
    </div>
  </div>

  <div style="margin-top: 16px; display: flex; align-items: center; gap: 12px;">
    <div style="font-size: 1rem; font-weight: 600;">Hugo de Lamo</div>
  </div>

  </div>
</div>

# 05 · Clasificador taxonómico de boletines oficiales

Este notebook implementa un clasificador multietiqueta de publicaciones de boletines oficiales españoles usando **Pydantic AI**. El problema es una aguja en un pajar: de ~65 000 publicaciones del Q1 2025, solo ~3,7 % son relevantes para el dominio ambiental-energético.

El clasificador responde cuatro preguntas por publicación:
1. **¿Es relevante?** — ¿Pertenece al universo de autorizaciones ambiental-energéticas?
2. **¿Qué procedimientos contiene?** — Lista multilabel: DIA, AAP, AAC, AAU, IIA, AAI, IAE, DUP.
3. **¿Qué tipo de acto es?** — Forma jurídica del documento (N1): resolución, anuncio, decreto…
4. **¿Qué tecnología menciona?** — Lista multilabel: fotovoltaica, eólica, hidrógeno…

---

## Estructura del notebook

| § | Sección | Contenido |
|---|---------|----------|
| **0** | **Setup** | Entorno, dependencias, modelo |
| **1** | **Schema de output** | `ClassifierOutput`, enums y reglas de negocio |
| **2** | **Ground truth** | Construcción del dataset etiquetado manualmente |
| **3** | **Agente clasificador** | System prompt, agent con Pydantic AI, run single |
| **4** | **Baseline** | Experimento 0 — claude-sonnet-4-6 sobre ground truth |
| **5** | **Experimentos de ablación** | Few-shot, `act_type` y campo `technologies` |
| **6** | **Modelos locales** | Experimento 4 — modelos 7B via LM Studio |
| **7** | **Análisis y conclusiones** | Comparativa de métricas, falsos positivos, siguientes pasos |

---

## §0. Setup

Cargamos las variables de entorno (la API key de Gemini vive en `.env`, nunca en el código) e importamos las librerías del proyecto.

In [ ]:
import os

import pandas as pd
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from pydantic_ai import Agent

load_dotenv()

In [ ]:
# Único punto de cambio para conectar otro proveedor.
# Para LM Studio (modelos locales): "openai:nombre-del-modelo" con base_url="http://127.0.0.1:1234/v1"
MODEL = "gemini-2.5-pro"

In [ ]:
api_key = os.getenv("GEMINI_API_KEY")
assert api_key, "GEMINI_API_KEY no encontrada — revisa el archivo .env"

print("Entorno listo.")
print(f"  Modelo:  {MODEL}")
print(f"  API key: {api_key[:8]}{'*' * (len(api_key) - 8)}")

---

## §1. Schema de output

El schema define el **contrato entre el LLM y el sistema**: qué campos devuelve el modelo, de qué tipo y bajo qué restricciones. Pydantic valida cada respuesta antes de que llegue al resto del código, forzando un retry automático si algo no cumple el schema.

`ClassifierOutput` tiene 6 campos:

| Campo | Tipo | Rol |
|-------|------|-----|
| `is_relevant` | `bool` | ¿Pertenece al dominio ambiental-energético? |
| `act_type` | `ActType` | Forma jurídica del acto (N1) — resolución, anuncio, decreto… |
| `procedures` | `list[ProcedureType]` | Procedimientos identificados (N2) — multilabel |
| `technologies` | `list[TechnologyType]` | Tecnologías mencionadas (N3) — multilabel, puede ser vacía |
| `confidence` | `float` | Confianza global entre 0.0 y 1.0 |
| `reasoning` | `str` | Justificación breve citando el texto que dispara cada etiqueta |

Un `@model_validator` impone los invariantes de negocio: `is_relevant=True` exige `procedures != []`; `is_relevant=False` exige ambas listas vacías.

In [ ]:
from enum import Enum

from pydantic import BaseModel, Field, model_validator


class ActType(str, Enum):
    RESOLUCION          = "resolución"
    ANUNCIO             = "anuncio"
    ORDEN               = "orden"
    DECRETO             = "decreto"
    ACUERDO             = "acuerdo"
    APROBACION          = "aprobación"
    INFORMACION_PUBLICA = "información_pública"
    CORRECCION_ERRORES  = "corrección_errores"
    NOTIFICACION        = "notificación"
    EXTRACTO            = "extracto"
    CONVENIO            = "convenio"
    SOLICITUD           = "solicitud"
    MODIFICACION        = "modificación"
    EDICTO              = "edicto"
    REAL_DECRETO        = "real_decreto"
    OTROS               = "otros"


class ProcedureType(str, Enum):
    DIA = "DIA"
    AAP = "AAP"
    AAC = "AAC"
    AAU = "AAU"
    IIA = "IIA"
    AAI = "AAI"
    IAE = "IAE"
    DUP = "DUP"


class TechnologyType(str, Enum):
    FOTOVOLTAICA     = "fotovoltaica"
    EOLICA           = "eólica"
    ALMACENAMIENTO   = "almacenamiento"
    HIBRIDACION      = "hibridación"
    HIDROELECTRICA   = "hidroeléctrica"
    BIOGAS_BIOMETANO = "biogás_biometano"
    BIOMASA          = "biomasa"
    HIDROGENO        = "hidrógeno"
    LINEA_ELECTRICA  = "línea_eléctrica"
    GAS_NATURAL      = "gas_natural"
    PETROLEO         = "petróleo"

In [ ]:
class ClassifierOutput(BaseModel):
    is_relevant: bool = Field(
        description=(
            "True si la publicación pertenece al universo ambiental-energético "
            "monitorizado. False para oposiciones, presupuestos, contratos, etc."
        )
    )
    act_type: ActType = Field(
        description=(
            "Tipo de acto administrativo (N1). Forma jurídica del documento "
            "inferida del primer token o estructura de la descripción."
        )
    )
    procedures: list[ProcedureType] = Field(
        description=(
            "Procedimientos identificados (N2). Lista vacía si is_relevant=False. "
            "Multilabel: una resolución puede otorgar AAP y AAC simultáneamente."
        )
    )
    technologies: list[TechnologyType] = Field(
        description=(
            "Tecnologías o sectores mencionados (N3). Puede ser lista vacía aunque "
            "is_relevant=True (ej. una AAP genérica sin tecnología explícita)."
        )
    )
    confidence: float = Field(
        description="Confianza global en la clasificación, entre 0.0 y 1.0.",
        ge=0.0,
        le=1.0,
    )
    reasoning: str = Field(
        description=(
            "Justificación breve. Citar el fragmento de texto que dispara cada "
            "etiqueta. Máximo 2 frases."
        )
    )

    @model_validator(mode="after")
    def check_invariants(self) -> "ClassifierOutput":
        if self.is_relevant and not self.procedures:
            raise ValueError("is_relevant=True requiere procedures != []")
        if not self.is_relevant:
            if self.procedures:
                raise ValueError("is_relevant=False con procedures != []")
            if self.technologies:
                raise ValueError("is_relevant=False con technologies != []")
        return self

In [ ]:
# Ejemplo válido: resolución de DIA con tecnología fotovoltaica
ejemplo_valido = ClassifierOutput(
    is_relevant=True,
    act_type=ActType.RESOLUCION,
    procedures=[ProcedureType.DIA],
    technologies=[TechnologyType.FOTOVOLTAICA],
    confidence=0.98,
    reasoning=(
        "'se formula la declaración de impacto ambiental' → DIA en resolución. "
        "'Planta Solar Fotovoltaica' → fotovoltaica."
    ),
)
print("Ejemplo válido:")
print(ejemplo_valido.model_dump_json(indent=2))

# Ejemplo que viola el invariante: is_relevant=False con procedures no vacío
print("\nViolación de invariante:")
try:
    ClassifierOutput(
        is_relevant=False,
        act_type=ActType.RESOLUCION,
        procedures=[ProcedureType.AAP],
        technologies=[],
        confidence=0.5,
        reasoning="Prueba de invariante.",
    )
except Exception as e:
    print(f"  ValidationError capturado → {e.errors()[0]['msg']}")